# Colab dots.ocr — the FAIR challenger test (full-page layout, CUDA)

dots.ocr (`rednote-hilab/dots.ocr`) emits document layout as JSON with **tables as HTML**
(carrying `colspan`/`rowspan`, which Surya 2 dropped). Apple-Silicon Stage A (PROJECT_LOG §2.85)
only completed ONE scored run — the hardest 124-DPI scan, off-label, on MPS (documented
correctness risk). Its best-case document type, born-digital, OOM'd before scoring. **This
notebook is the fair test:** CUDA (numerically correct, unlike MPS), full resolution, dots.ocr's
DESIGNED full-page mode, on every document class.

Emits `predictions.json = {"<image.png>": "<raw model output>"}`, scored locally:

```bash
uv run python scripts/score_dots_predictions.py --predictions predictions.json
```

**Steps:**
1. Runtime ▸ Change runtime type ▸ **T4 GPU**.
2. Run the install cell, then the **folder** cell — it creates `pages/`.
3. In the Colab **Files** pane (left sidebar), open `pages/` and **upload your PNGs** there
   (from `eval/datasets/real/`, `eval/datasets/budget_textlayer/`, `eval/datasets/moc_gas/`).
   Re-run the folder cell to confirm they are visible.
4. Run the model, prompt, and inference cells → the last cell downloads `predictions.json`.

Privacy: these are the financial PNGs; Colab (open model) is the path chosen over a commercial
cloud API — and every eval doc here is an already-published public bulletin.

> **T4 note:** Turing (compute 7.5) has **no hardware bfloat16** and no flash-attention-2. This
> notebook auto-selects **float16** on such GPUs and uses CUDA **sdpa** attention — correct and
> memory-efficient. On a Pro L4/A100 (Ampere+) it uses bfloat16 automatically.

In [ ]:
!pip install -q -U "transformers>=4.51,<4.58" accelerate torchvision pillow

In [ ]:
# --- Upload folder ---------------------------------------------------------
# Creates pages/. Upload your .png files INTO it via the Colab Files pane
# (left sidebar ▸ Files ▸ pages ▸ Upload). Subfolders are fine. Re-run this cell
# any time to confirm what it sees. Filenames become the prediction keys, which
# the local scorer maps by stem — so a bare filename is what it expects.
import os
from pathlib import Path

INPUT_DIR = Path("pages")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Upload PNGs into this folder (Files pane ▸ pages ▸ Upload):")
print(" ", INPUT_DIR.resolve(), "\n")

png_paths = sorted((p for p in INPUT_DIR.rglob("*")
                    if p.is_file() and p.suffix.lower() == ".png"),
                   key=lambda p: p.name)
print(f"found {len(png_paths)} PNG file(s):")
for p in png_paths:
    print("  -", p.relative_to(INPUT_DIR))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoImageProcessor, AutoConfig

REPO = "rednote-hilab/dots.ocr"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA. Runtime ▸ Change runtime type ▸ T4 GPU, then rerun.")

# T4 (Turing) has NO hardware bfloat16 — using bf16 there is slow or wrong. Pick
# bf16 only when the GPU actually supports it (Ampere+), else float16. The model's
# bf16 weights cast to fp16 cleanly.
try:
    BF16_OK = torch.cuda.is_bf16_supported()
except Exception:
    BF16_OK = False
DTYPE = torch.bfloat16 if BF16_OK else torch.float16

# Stage-A lesson: the VISION tower has its own attn setting, defaulting to
# flash_attention_2. On any non-flash GPU that must be sdpa, or it silently falls
# back to eager and materialises the full patch-attention matrix (OOM).
cfg = AutoConfig.from_pretrained(REPO, trust_remote_code=True)
cfg.attn_implementation = "sdpa"
if getattr(cfg, "vision_config", None) is not None:
    cfg.vision_config.attn_implementation = "sdpa"

# torch_dtype (not dtype): accepted across the whole transformers>=4.51,<4.58 pin;
# `dtype` only exists on 4.56+ and would error on an older resolve.
model = AutoModelForCausalLM.from_pretrained(
    REPO, config=cfg, trust_remote_code=True,
    torch_dtype=DTYPE, attn_implementation="sdpa", low_cpu_mem_usage=True,
).to("cuda").eval()

# AutoProcessor is unusable under transformers 4.5x: DotsVLProcessor subclasses
# Qwen2_5_VLProcessor with a 3-arg __init__, but video_processor is now required.
# Build the two real pieces directly.
tok = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)
improc = AutoImageProcessor.from_pretrained(REPO, trust_remote_code=True)
print("loaded", REPO, "| dtype", DTYPE)

In [ ]:
# dots.ocr's OFFICIAL full-page prompt (prompt_layout_all_en). Tables as HTML.
PROMPT = (
    "Please output the layout information from the PDF image, including each layout "
    "element's bbox, its category, and the corresponding text content within the bbox.\n\n"
    "1. Bbox format: [x1, y1, x2, y2]\n\n"
    "2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', "
    "'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', "
    "'Title'].\n\n"
    "3. Text Extraction & Formatting Rules:\n"
    "    - Picture: For the 'Picture' category, the text field should be omitted.\n"
    "    - Formula: Format its text as LaTeX.\n"
    "    - Table: Format its text as HTML.\n"
    "    - All Others (Text, Title, etc.): Format their text as Markdown.\n\n"
    "4. Constraints:\n"
    "    - The output text must be the original text from the image, with no translation.\n"
    "    - All layout elements must be sorted according to human reading order.\n\n"
    "5. Final Output: The entire output must be a single JSON object.\n"
)

In [ ]:
import json, os, time
from pathlib import Path
from PIL import Image
Image.MAX_IMAGE_PIXELS = None  # large scans must not trip the decompression-bomb guard

INPUT_DIR = Path("pages")
# Full resolution: T4 has 16GB, so we do NOT downscale the way MPS forced us to.
# HTML for a dense table (34x16 = 544 cells) is long; too small a cap truncates
# mid-table (a Stage-A failure mode). Lower to 8192 if the T4 OOMs.
MAX_NEW_TOKENS = 12000
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
MERGE = int(getattr(improc, "merge_size", 2))

paths = sorted((p for p in INPUT_DIR.rglob("*")
                if p.is_file() and p.suffix.lower() == ".png"),
               key=lambda p: str(p.relative_to(INPUT_DIR)))
if not paths:
    raise FileNotFoundError(f"No .png in {INPUT_DIR}/. Upload via the Files pane, then rerun.")

def run_page(path):
    img = Image.open(path).convert("RGB")
    vis = improc(images=[img], return_tensors="pt")
    grid = vis["image_grid_thw"]
    # Qwen2-VL packing: image collapses to grid patches, then merge_size**2 patches
    # fuse into one token, so the <|imgpad|> placeholder repeats exactly that many times.
    n_img = int(grid.prod().item()) // (MERGE ** 2)
    # dots.ocr's OWN chat format, NOT Qwen's <|im_start|> — the wrong one emits EOS instantly.
    # No spaces inside the special tokens.
    prompt = ("<|user|><|img|>" + "<|imgpad|>" * n_img + "<|endofimg|>"
              + PROMPT + "<|endofuser|><|assistant|>")
    # add_special_tokens=False: the full chat frame is already in the string; let the
    # tokenizer add nothing of its own, or a stray BOS shifts the image alignment.
    enc = tok([prompt], return_tensors="pt", add_special_tokens=False)
    inputs = {
        "input_ids": enc.input_ids.to("cuda"),
        "attention_mask": enc.attention_mask.to("cuda"),
        "pixel_values": vis["pixel_values"].to("cuda", dtype=DTYPE),
        "image_grid_thw": grid.to("cuda"),
    }
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=PAD_ID)
    trimmed = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(trimmed, skip_special_tokens=True).strip()
    n_new = int(trimmed.shape[0])
    flag = "  TRUNCATED(hit cap)" if n_new >= MAX_NEW_TOKENS else ""
    print(f"{path.name}: {img.size} -> {n_img} img tokens, {n_new} new tokens, "
          f"{time.perf_counter()-t0:.0f}s{flag}")
    del inputs, out
    torch.cuda.empty_cache()
    return raw

preds = {}
for path in paths:
    key = path.name  # bare filename; the scorer maps by stem
    if key in preds:
        # Two uploads share a basename — fall back to the relative path so neither
        # silently overwrites the other. The scorer's stem lookup still resolves it.
        key = str(path.relative_to(INPUT_DIR)).replace(os.sep, "/")
        print(f"WARNING: duplicate filename; keyed as '{key}'")
    preds[key] = run_page(path)

with open("predictions.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)
print("wrote predictions.json with", len(preds), "predictions")

In [ ]:
from google.colab import files
files.download("predictions.json")